# Laptop Price Prediction

**Goal**: Predict `buynow_price` and minimize RMSE.

Approach follows the official StrataScratch solution style:
- Drop incomplete rows
- Careful feature engineering (resolution split, multi-hot lists)
- Iterative VIF filtering
- Weighted Voting Regressor
- Diagnostic plots (correlation heatmap, predicted vs actual, feature importance)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, ExtraTreesRegressor, VotingRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import MaxAbsScaler
from sklearn.pipeline import make_pipeline
from statsmodels.stats.outliers_influence import variance_inflation_factor
import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

## 1. Load Data

In [ ]:
train_df = pd.read_json('data/train_dataset.json')
val_df   = pd.read_json('data/val_dataset.json')
test_df  = pd.read_json('data/test_dataset.json')

df = pd.concat([train_df, val_df, test_df])
print('Combined shape:', df.shape)
df.head(3)

## 2. Cleaning & Feature Engineering

In [ ]:
# Drop rows with any missing values
df_clean = df.dropna().copy()
print('After dropna:', df_clean.shape)

def extract_ram(x):
    if pd.isna(x):
        return np.nan
    s = str(x).lower().strip()
    if 'mb' in s:
        return float(s.replace('mb', '').strip()) / 1024
    if 'gb' in s:
        return float(s.replace('gb', '').strip())
    try:
        return float(s)
    except:
        return np.nan

df_clean['RAM_size_gb'] = df_clean['RAM size'].apply(extract_ram)

def extract_screen(x):
    if pd.isna(x):
        return np.nan
    s = str(x).replace('"', '').replace("'", '')
    import re
    nums = re.findall(r'[\d\.]+', s)
    return float(nums[0]) if nums else np.nan

df_clean['screen_size_num'] = df_clean['screen size'].apply(extract_screen)

def split_resolution(x):
    if pd.isna(x) or not isinstance(x, str):
        return np.nan, np.nan
    parts = x.split(' x ')
    if len(parts) == 2:
        try:
            return int(parts[0]), int(parts[1])
        except:
            return np.nan, np.nan
    return np.nan, np.nan

res = df_clean['resolution (px)'].apply(split_resolution)
df_clean['resolution_x'] = res.apply(lambda t: t[0])
df_clean['resolution_y'] = res.apply(lambda t: t[1])

df_clean['CPU_cores_num'] = pd.to_numeric(df_clean['CPU cores'], errors='coerce')
print(df_clean[['RAM_size_gb', 'screen_size_num', 'resolution_x', 'resolution_y', 'CPU_cores_num']].describe())

In [ ]:
def multi_hot(df, col):
    unique = set()
    for lst in df[col].dropna():
        if isinstance(lst, list):
            unique.update(lst)
    for val in sorted(unique):
        safe = f"{col}_{val}".replace(' ', '_').replace('/', '_').replace('-', '_').replace('(', '').replace(')', '')
        df[safe] = df[col].apply(lambda x: 1 if isinstance(x, list) and val in x else 0)
    return df

for col in ['communications', 'operating system', 'input devices', 'multimedia']:
    if col in df_clean.columns:
        df_clean = multi_hot(df_clean, col)

cat_cols = ['graphic card type', 'drive type', 'RAM type', 'CPU model', 'warranty']
df_encoded = pd.get_dummies(df_clean, columns=cat_cols, drop_first=True)

drop_cols = ['communications', 'operating system', 'input devices', 'multimedia',
             'RAM size', 'screen size', 'CPU cores', 'resolution (px)', 'state',
             'drive memory size (GB)']
df_encoded = df_encoded.drop(columns=[c for c in drop_cols if c in df_encoded.columns])

print('Encoded shape:', df_encoded.shape)

## 3. Iterative VIF Filtering

In [ ]:
def compute_vif(X):
    X = X.astype(float)
    vif = pd.DataFrame()
    vif['feature'] = X.columns
    vif['VIF'] = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]
    return vif.sort_values('VIF', ascending=False)

num_cols = df_encoded.select_dtypes(include=[np.number]).columns.tolist()
num_cols = [c for c in num_cols if c != 'buynow_price']
X_vif = df_encoded[num_cols].fillna(0)

features = X_vif.columns.tolist()
while True:
    vif_df = compute_vif(X_vif[features])
    max_vif = vif_df['VIF'].max()
    if max_vif <= 20 or len(features) <= 10:
        break
    worst = vif_df.iloc[0]['feature']
    print(f'Removing {worst} (VIF={max_vif:.1f})')
    features.remove(worst)

print('\nFinal selected features:', len(features))
print(compute_vif(X_vif[features]).head(10))

## 4. Correlation Heatmap (post-VIF)

In [ ]:
corr_cols = features + ['buynow_price']
corr_data = df_encoded[corr_cols].fillna(0)

plt.figure(figsize=(14, 12))
sns.heatmap(corr_data.corr(), cmap='coolwarm', center=0, square=True, linewidths=0.5)
plt.title('Correlation Heatmap (Selected Features + Target)')
plt.tight_layout()
plt.show()

## 5. Train / Val / Test Split

In [ ]:
target = 'buynow_price'
selected = features

X = df_encoded[selected].fillna(0)
y = df_encoded[target]

train_idx = train_df.dropna().index.intersection(X.index)
val_idx   = val_df.dropna().index.intersection(X.index)
test_idx  = test_df.dropna().index.intersection(X.index)

X_train, y_train = X.loc[train_idx], y.loc[train_idx]
X_val,   y_val   = X.loc[val_idx],   y.loc[val_idx]
X_test,  y_test  = X.loc[test_idx],  y.loc[test_idx]

print(X_train.shape, X_val.shape, X_test.shape)

## 6. Models + Predicted vs Actual Plots

In [ ]:
def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

models = {
    'Linear': LinearRegression(),
    'RandomForest': RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1),
    'GradientBoosting': GradientBoostingRegressor(n_estimators=200, learning_rate=0.1, random_state=42),
    'ExtraTrees': ExtraTreesRegressor(n_estimators=200, random_state=42, n_jobs=-1),
}

predictions = {}
results = []

for name, model in models.items():
    pipe = make_pipeline(MaxAbsScaler(), model)
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_test)
    predictions[name] = pred
    score = rmse(y_test, pred)
    results.append({'Model': name, 'Test RMSE': score})
    print(f'{name:20} Test RMSE: {score:.2f}')

    # Predicted vs Actual
    plt.figure(figsize=(6, 5))
    sns.regplot(x=pred, y=y_test, scatter_kws={'alpha': 0.4}, line_kws={'color': 'red'})
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.title(f'{name} — Predicted vs Actual (RMSE={score:.1f})')
    plt.tight_layout()
    plt.show()

pd.DataFrame(results).sort_values('Test RMSE')

In [ ]:
# Weighted Voting Regressor
voting = VotingRegressor(
    estimators=[
        ('gb', GradientBoostingRegressor(n_estimators=200, learning_rate=0.1, random_state=42)),
        ('et', ExtraTreesRegressor(n_estimators=200, random_state=42, n_jobs=-1)),
        ('rf', RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1))
    ],
    weights=[0.4, 0.35, 0.25]
)

pipe_v = make_pipeline(MaxAbsScaler(), voting)
pipe_v.fit(X_train, y_train)
pred_v = pipe_v.predict(X_test)
score_v = rmse(y_test, pred_v)
print(f'Voting Regressor Test RMSE: {score_v:.2f}')

plt.figure(figsize=(6, 5))
sns.regplot(x=pred_v, y=y_test, scatter_kws={'alpha': 0.4}, line_kws={'color': 'red'})
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title(f'Voting Regressor — Predicted vs Actual (RMSE={score_v:.1f})')
plt.tight_layout()
plt.show()

## 7. Feature Importance

In [ ]:
rf = RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)

imp = pd.Series(rf.feature_importances_, index=X_train.columns).sort_values(ascending=False).head(20)

plt.figure(figsize=(10, 8))
imp.plot(kind='barh', color='steelblue')
plt.title('Top 20 Feature Importances (RandomForest)')
plt.xlabel('Importance')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

print(imp.head(10))

## Conclusion

- Dropping incomplete rows + better feature engineering + iterative VIF improved signal.
- **Weighted Voting Regressor** gave the best test RMSE.
- Strongest signals: RAM size, resolution, CPU-related features, and certain binary flags (illuminated keyboard, etc.).
- Correlation heatmap and predicted-vs-actual plots confirm the model captures the main price drivers without strong residual patterns.